## Faster C-NN
___

In [ ]:
from pathlib import Path
import torch
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torch.utils.data import Dataset, DataLoader
import pandas as pd

# === 1. Definición del Dataset Personalizado ===
class DatasetPeces(Dataset):
    def __init__(self, df_anotaciones, carpeta_imagenes):
        """
        df_anotaciones debe contener columnas: ['archivo', 'x1', 'y1', 'x2', 'y2', 'clase_id']
        """
        self.df = df_anotaciones
        self.carpeta = Path(carpeta_imagenes)
        self.archivos = df_anotaciones['archivo'].unique()

    def __len__(self):
        return len(self.archivos)

    def __getitem__(self, idx):
        nombre_archivo = self.archivos[idx]
        ruta_img = self.carpeta / nombre_archivo
        
        # Cargar imagen y convertir a tensor [0, 1]
        img = torchvision.io.read_image(str(ruta_img)).float() / 255.0

        # Filtrar las cajas pertenecientes a esta imagen
        filas = self.df[self.df['archivo'] == nombre_archivo]
        cajas = filas[['x1', 'y1', 'x2', 'y2']].values
        etiquetas = filas['clase_id'].values

        target = {
            "boxes": torch.tensor(cajas, dtype=torch.float32),
            "labels": torch.tensor(etiquetas, dtype=torch.int64),
        }

        return img, target

def collate_fn(batch):
    return tuple(zip(*batch))


# === 2. Configuración e Inicialización ===
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Supongamos 1 clase de pez + 1 fondo = 2 clases
NUM_CLASES = 2 

# Cargar modelo y reemplazar la cabeza de predicción
pesos = torchvision.models.detection.FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT
model = torchvision.models.detection.fasterrcnn_resnet50_fpn_v2(weights=pesos)

in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, NUM_CLASES)

# Congelamos el backbone para entrenar rápidamente solo el predictor y RPN
for param in model.backbone.parameters():
    param.requires_grad = False

model.to(device)

# Seleccionar únicamente los parámetros que requieren gradiente
parametros_entrenables = [p for p in model.parameters() if p.requires_grad]

optimizer = torch.optim.SGD(
    parametros_entrenables, 
    lr=0.005, 
    momentum=0.9, 
    weight_decay=0.0005
)

# Cargar anotaciones previas (ejemplo en CSV) y crear DataLoader
# df_etiquetas = pd.read_csv("mis_anotaciones_entrenamiento.csv")
# dataset = DatasetPeces(df_etiquetas, "data/FjordFish/images/train")
# dataloader = DataLoader(dataset, batch_size=2, shuffle=True, collate_fn=collate_fn)


# === 3. Bucle de Entrenamiento (Training Loop) ===
EPOCHS = 10

model.train() # Activa el modo de entrenamiento

for epoch in range(EPOCHS):
    pérdida_acumulada = 0.0

    for imagenes, targets in dataloader:
        imagenes = [img.to(device) for img in imagenes]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        # En modo train, el modelo retorna un diccionario de pérdidas
        loss_dict = model(imagenes, targets)
        
        # Sumamos todas las pérdidas (clasificación + regresión de cajas)
        total_loss = sum(loss for loss in loss_dict.values())

        # Paso de optimización
        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()

        pérdida_acumulada += total_loss.item()

    promedio_epoca = pérdida_acumulada / len(dataloader)
    print(f"Época [{epoch + 1}/{EPOCHS}] - Pérdida Total: {promedio_epoca:.4f}")

# === 4. Guardar los pesos entrenados ===
torch.save(model.state_dict(), "fasterrcnn_pesos_finetuned.pth")

Se encontraron 2086 imágenes en 'data/FjordFish/images/train'
Procesando: 1.jpg


/home/cristopher/Documentos/Workspace/Deep Learning/tarea 2/tarea_2/lib/python3.12/site-packages/torchvision/utils.py:233: UserWarning: Argument 'font_size' will be ignored since 'font' is not set.
  warnings.warn("Argument 'font_size' will be ignored since 'font' is not set.")


Procesando: 2.jpg
Procesando: 3.jpg
Procesando: 4.jpg


KeyboardInterrupt: 

## YOLO

___

In [ ]:
import os
import re
from pathlib import Path
import pandas as pd
from ultralytics import YOLO

#=== Variables Varias ===
CARPETA_ENTRADA = "data/FjordFish/images/train"
CARPETA_SALIDA  = Path("resultados_yolo")
ARCHIVO_CSV     = "detecciones_yolo.csv"
EXTENSION = ".jpg"
UMBRAL_CONFIANZA    = 0.7

os.makedirs(CARPETA_SALIDA, exist_ok=True)

#Cargamos uno de los tantos modelos de yolo que hay ('yolov8s.pt' o 'yolov8m.pt') /////////////////////////////EXPLICAR PQ USAMOS ESE ///////////////////////
model = YOLO("yolov8n.pt") 


carpeta = Path(CARPETA_ENTRADA)
rutas = []

#Misma lógica que en el modelo anterior
for archivo in carpeta.iterdir():
    #Obtenemos solo las imagenes de la carpeta, asi evitamos obtener los archivos ocultoas
    if archivo.suffix == EXTENSION:
        rutas.append(archivo)

def extraer_numero(path):
    numeros = re.findall(r'\d+', path.name)
    if numeros != 0:
        return int(numeros[0])
    else:
        return 0

#Ordenamos en base a los numeros
rutas.sort(key=extraer_numero)
print(f"Se encontraron {len(rutas)} imágenes en '{CARPETA_ENTRADA}'")


#=== procesamos las imagenes y guardamos su información ===
registros = []

for ruta in rutas:
    print(f"Procesando: {ruta.name}")

    # generamos las inferencias con verbose false, para no ver tanto log
    results = model.predict(source=str(ruta), conf=UMBRAL_CONFIANZA, verbose=False)[0]

    # YOLO dibuja y guarda la imagen anotada automáticamente en la carpeta de destino
    results.save(filename=str(CARPETA_SALIDA / ruta.name))

    # Extraer las detecciones obtenidas
    boxes = results.boxes

    # Guardamos la info de las imagenes sin objetos, como la que tienen objetos
    if len(boxes) == 0:
        registros.append({
            "archivo": ruta.name,
            "etiqueta": None,
            "confianza": None,
            "x1": None, "y1": None, "x2": None, "y2": None,
        })
    else:
        print("Se encontro un objeto")
        # Convertir datos de tensores a arrays de Python/NumPy
        xyxy_list = boxes.xyxy.cpu().numpy()
        conf_list = boxes.conf.cpu().numpy()     
        cls_list  = boxes.cls.cpu().numpy()      

        for box, conf, cls_id in zip(xyxy_list, conf_list, cls_list):
            x1, y1, x2, y2 = box
            
            # Traducimos el id a su nombre
            nombre_clase = model.names[int(cls_id)]  

            registros.append({
                "archivo": ruta.name,
                "etiqueta": nombre_clase,
                "confianza": round(float(conf), 4),
                "x1": round(float(x1), 1),
                "y1": round(float(y1), 1),
                "x2": round(float(x2), 1),
                "y2": round(float(y2), 1),
            })

#=== Creamos el dt y lo exportamos ===
df = pd.DataFrame(registros)
df.to_csv(ARCHIVO_CSV, index=False, encoding="utf-8")

print("\n===== RESUMEN YOLO =====")
print(f"Total de filas (detecciones): {len(df)}")
print(f"Imágenes procesadas: {df['archivo'].nunique()}")
print(f"\nPrimeras filas:\n{df.head(10)}")
print(f"\nConteo por etiqueta:\n{df['etiqueta'].value_counts()}")
print(f"\nResultados guardados en: {ARCHIVO_CSV}")
print(f"Imágenes anotadas en: {CARPETA_SALIDA}/")

Se encontraron 2086 imágenes en 'data/FjordFish/images/train'
Procesando: 1.jpg
Procesando: 2.jpg
Procesando: 3.jpg
Procesando: 4.jpg
Procesando: 5.jpg
Procesando: 6.jpg
Procesando: 7.jpg
Procesando: 8.jpg
Procesando: 9.jpg
Procesando: 10.jpg
Procesando: 12.jpg
Procesando: 13.jpg
Procesando: 14.jpg
Procesando: 17.jpg
Procesando: 19.jpg
Procesando: 21.jpg
Procesando: 23.jpg
Procesando: 24.jpg
Procesando: 25.jpg
Procesando: 26.jpg
Procesando: 30.jpg
Procesando: 32.jpg
Procesando: 34.jpg
Procesando: 35.jpg
Procesando: 36.jpg
Procesando: 37.jpg
Procesando: 39.jpg
Procesando: 41.jpg
Procesando: 42.jpg
Procesando: 43.jpg
Procesando: 44.jpg
Procesando: 45.jpg
Procesando: 47.jpg
Procesando: 49.jpg
Se encontro un objeto
Procesando: 50.jpg
Procesando: 51.jpg
Procesando: 52.jpg
Procesando: 54.jpg
Procesando: 55.jpg
Procesando: 56.jpg
Procesando: 57.jpg
Procesando: 58.jpg
Procesando: 59.jpg
Procesando: 60.jpg
Procesando: 61.jpg
Procesando: 62.jpg
Procesando: 63.jpg
Procesando: 64.jpg
Procesando: 6